# Uber Airtable:   Read Customer and Driver info from a database, then make an "optimal" assignment.

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2024, University of Chicago 

---

In practice, at least some if not all of your data for your model is contained in a database.  We will now get our Uber Driver and Customer data from an Airtable Base.  

A primary goal here, is that our database can and will change, but our optimization code should not!


### Access the Uber base 

#### Do the following:

1. You need to share my Uber database within your AirTable account by, [clicking here](https://airtable.com/invite/l?inviteId=invaS7vygblNcohv1&inviteToken=3791960bd836e2ffd30b5437c7c7112aa8ff3a04a6b9f9becb7d1738113ad0bd&utm_medium=email&utm_source=product_team&utm_content=transactional-alerts)

1. Click your Account in the upper right corner and select "Developer Hub"

1. Click "..." on far right by your account token and select "Edit"

1. Then click "Add a base" and select "Uber"


### Installing the pyairtable python wrapper

If you are on your local machine and have not installed this wrapper then uncomment and execute the following line

In [17]:
# === SETUP ===
import pulp
import os
from pprint import pprint

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

In [18]:
from pyairtable import Api
from pprint import pprint

### pyairtable
We are using the package [pyairtable](https://pyairtable.readthedocs.io/en/stable/getting-started.html) as an easy way to interface with airtable from python.

In [6]:
AIRTABLE_API_TOKEN = 'patDlJ0vTK7ZXbVpL.63bf7ff0db1d1e8d2f932285d08e17ac3a11bd23395efb4910d6cdfacf91581b'
AIRTABLE_BASE_ID = 'app5zooNUJ7gOtZQz'   # The ID for the Uber Base

CUSTOMER_TABLE_ID = "Customer" # name of the customer table in the Uber base
DRIVER_TABLE_ID = "Driver" # name of the driver table in the Uber base

# create a new client configured with your api token
api = Api(AIRTABLE_API_TOKEN)

# use the client created in the prior cell to get all of the records in the TestTable table
customer_table = api.table(AIRTABLE_BASE_ID, CUSTOMER_TABLE_ID)
driver_table = api.table(AIRTABLE_BASE_ID, DRIVER_TABLE_ID)

We will now fill in our data structures that we hard-coded before, with information from our Uber database.

First, we explore, by iterating over each row in the cutomer table using our familiar `for` statement.

We print out each row, which we see is simply a Python dictionary ... we are ready to go!

In [23]:
pprint(customer_table.all())
print('\n')
pprint(driver_table.all())

[{'createdTime': '2019-08-24T18:07:15.000Z',
  'fields': {'Address': 'Hyatt Atlanta, Atlanta, GA',
             'Name': 'Sean Hernandez',
             'Num': 'C331'},
  'id': 'rec0jKiObW7a16CUB'},
 {'createdTime': '2019-08-24T18:07:15.000Z',
  'fields': {'Address': 'Taco Mac, Virginia Highlands, Atlanta, GA',
             'Name': 'Anuj Patel',
             'Num': 'C934'},
  'id': 'recJCIS5tRcEqSkHs'},
 {'createdTime': '2023-04-08T16:52:09.000Z',
  'fields': {'Address': 'Georgia Aquarium, Atlanta, GA',
             'Name': 'Marc Adams',
             'Num': 'C17'},
  'id': 'recMuNGdLHWa1CS0v'},
 {'createdTime': '2019-08-24T18:07:15.000Z',
  'fields': {'Address': '1400 Briarcliff Rd NE, Apt 621, Atlanta, GA 30306',
             'Name': 'Tracy Higgins',
             'Num': 'C11'},
  'id': 'recZtIdCYXLVNzY0c'},
 {'createdTime': '2023-04-08T16:52:32.000Z',
  'fields': {'Address': 'Georgia Tech, Atlanta, GA',
             'Name': 'Cindy Sharm',
             'Num': 'C987'},
  'id': 'recmL7nNQ3

### Now we'll create our list of drivers and customers

In [15]:
customers = []
for customer in customer_table.all():
    customers.append(customer['fields']['Num'])

print(customers)

['C331', 'C934', 'C17', 'C11', 'C987']


In [16]:
drivers = []
for driver in driver_table.all():
    drivers.append(driver['fields']['Num'])

print(drivers)

['D922', 'D88', 'D32', 'D8712', 'D192']


### Let's now read in the driver locations and the customer addresses

In [35]:
customer_addresses = {}
for customer in customer_table.all():
    c = customer['fields'] # Shorten our lines of code a little
    customer_addresses.update( [ (c['Num'], c['Address']) ] )

pprint(customer_addresses)

{'C11': '1400 Briarcliff Rd NE, Apt 621, Atlanta, GA 30306',
 'C17': 'Georgia Aquarium, Atlanta, GA',
 'C331': 'Hyatt Atlanta, Atlanta, GA',
 'C934': 'Taco Mac, Virginia Highlands, Atlanta, GA',
 'C987': 'Georgia Tech, Atlanta, GA'}


In [36]:
driver_locations = {}
for driver in driver_table.all():
    d = driver['fields'] # Shorten our lines of code a little
    driver_locations.update( [ ( d['Num'],                                # Keys
                                [float(d['Lat']), float(d['Lng'])] ) ] ) # Values: [Lat, Lng] pairs

pprint(driver_locations)

{'D192': [33.7711, -84.3862],
 'D32': [33.793711, -84.317408],
 'D8712': [33.77845, -84.400825],
 'D88': [33.77655, -84.320082],
 'D922': [33.775306, -84.396123]}


###  Now we can copy paste our code from Uber Google to solve the model

The following code blocks now repeat code we have written before.  That is, now that we have used Airtable to fill in our data structures, we are back to where we were, ready to ask Google for travel times, and build our model with PuLP

### Use Google API for distances between each pair of customers and drivers

In [37]:
# import our google_functions, make sure a copy of it is located in this same folder
from google_functions import *

In [38]:
googlemap = create_googlemaps_object()
travel_times = {}
for driver in drivers:
    for customer in customers:
        google_time = duration_in_traffic(googlemap, driver_locations[driver], customer_addresses[customer])
        print(f"Time from {driver} to {customer} is {google_time}")
        travel_times[(driver,customer)]=google_time 
pprint(travel_times)

Time from D922 to C331 is 772
Time from D922 to C934 is 964
Time from D922 to C17 is 594
Time from D922 to C11 is 1435
Time from D922 to C987 is 286
Time from D88 to C331 is 1222
Time from D88 to C934 is 827
Time from D88 to C17 is 1469
Time from D88 to C11 is 560
Time from D88 to C987 is 1311
Time from D32 to C331 is 1412
Time from D32 to C934 is 932
Time from D32 to C17 is 1661
Time from D32 to C11 is 530
Time from D32 to C987 is 1509
Time from D8712 to C331 is 873
Time from D8712 to C934 is 1085
Time from D8712 to C17 is 514
Time from D8712 to C11 is 1565
Time from D8712 to C987 is 114
Time from D192 to C331 is 378
Time from D192 to C934 is 559
Time from D192 to C17 is 459
Time from D192 to C11 is 1045
Time from D192 to C987 is 252
{('D192', 'C11'): 1045,
 ('D192', 'C17'): 459,
 ('D192', 'C331'): 378,
 ('D192', 'C934'): 559,
 ('D192', 'C987'): 252,
 ('D32', 'C11'): 530,
 ('D32', 'C17'): 1661,
 ('D32', 'C331'): 1412,
 ('D32', 'C934'): 932,
 ('D32', 'C987'): 1509,
 ('D8712', 'C11'): 1

In [39]:
from pulp import *
model = LpProblem("Uber_Assignment",LpMinimize)

In [40]:
X= {} 
for driver in drivers: 
    for customer in customers:
        X[(driver,customer)] = LpVariable(f"X_{driver}_{customer}", cat='Binary')
print(f'Flow Variables X: {X}')

Flow Variables X: {('D922', 'C331'): X_D922_C331, ('D922', 'C934'): X_D922_C934, ('D922', 'C17'): X_D922_C17, ('D922', 'C11'): X_D922_C11, ('D922', 'C987'): X_D922_C987, ('D88', 'C331'): X_D88_C331, ('D88', 'C934'): X_D88_C934, ('D88', 'C17'): X_D88_C17, ('D88', 'C11'): X_D88_C11, ('D88', 'C987'): X_D88_C987, ('D32', 'C331'): X_D32_C331, ('D32', 'C934'): X_D32_C934, ('D32', 'C17'): X_D32_C17, ('D32', 'C11'): X_D32_C11, ('D32', 'C987'): X_D32_C987, ('D8712', 'C331'): X_D8712_C331, ('D8712', 'C934'): X_D8712_C934, ('D8712', 'C17'): X_D8712_C17, ('D8712', 'C11'): X_D8712_C11, ('D8712', 'C987'): X_D8712_C987, ('D192', 'C331'): X_D192_C331, ('D192', 'C934'): X_D192_C934, ('D192', 'C17'): X_D192_C17, ('D192', 'C11'): X_D192_C11, ('D192', 'C987'): X_D192_C987}


In [41]:
# Let's loop through OUR variable names, like 'C1_D2', 
# from those we can access the distance and true PuLP variable objects
obj = ''
for driver in drivers: 
    for customer in customers: 
        obj += travel_times[(driver,customer)] * X[(driver,customer)]
model += obj, "Cost of Customer Driver Assignment"
print(model)

Uber_Assignment:
MINIMIZE
1045*X_D192_C11 + 459*X_D192_C17 + 378*X_D192_C331 + 559*X_D192_C934 + 252*X_D192_C987 + 530*X_D32_C11 + 1661*X_D32_C17 + 1412*X_D32_C331 + 932*X_D32_C934 + 1509*X_D32_C987 + 1565*X_D8712_C11 + 514*X_D8712_C17 + 873*X_D8712_C331 + 1085*X_D8712_C934 + 114*X_D8712_C987 + 560*X_D88_C11 + 1469*X_D88_C17 + 1222*X_D88_C331 + 827*X_D88_C934 + 1311*X_D88_C987 + 1435*X_D922_C11 + 594*X_D922_C17 + 772*X_D922_C331 + 964*X_D922_C934 + 286*X_D922_C987 + 0
VARIABLES
0 <= X_D192_C11 <= 1 Integer
0 <= X_D192_C17 <= 1 Integer
0 <= X_D192_C331 <= 1 Integer
0 <= X_D192_C934 <= 1 Integer
0 <= X_D192_C987 <= 1 Integer
0 <= X_D32_C11 <= 1 Integer
0 <= X_D32_C17 <= 1 Integer
0 <= X_D32_C331 <= 1 Integer
0 <= X_D32_C934 <= 1 Integer
0 <= X_D32_C987 <= 1 Integer
0 <= X_D8712_C11 <= 1 Integer
0 <= X_D8712_C17 <= 1 Integer
0 <= X_D8712_C331 <= 1 Integer
0 <= X_D8712_C934 <= 1 Integer
0 <= X_D8712_C987 <= 1 Integer
0 <= X_D88_C11 <= 1 Integer
0 <= X_D88_C17 <= 1 Integer
0 <= X_D88_C331 <

In [42]:
#  Or, of course, let's use Python loops!!
# First the driver constraints
for driver in drivers:
    const = ''
    for customer in customers:
        const += X[(driver,customer)]
    model += const == 1, f"driver_{driver}"
# Next let's do the customer constraints
for customer in customers:
    const = ''
    for driver in drivers:
        const += X[(driver,customer)]
    model += const == 1, f"customer_{customer}"


In [43]:
print(model)

Uber_Assignment:
MINIMIZE
1045*X_D192_C11 + 459*X_D192_C17 + 378*X_D192_C331 + 559*X_D192_C934 + 252*X_D192_C987 + 530*X_D32_C11 + 1661*X_D32_C17 + 1412*X_D32_C331 + 932*X_D32_C934 + 1509*X_D32_C987 + 1565*X_D8712_C11 + 514*X_D8712_C17 + 873*X_D8712_C331 + 1085*X_D8712_C934 + 114*X_D8712_C987 + 560*X_D88_C11 + 1469*X_D88_C17 + 1222*X_D88_C331 + 827*X_D88_C934 + 1311*X_D88_C987 + 1435*X_D922_C11 + 594*X_D922_C17 + 772*X_D922_C331 + 964*X_D922_C934 + 286*X_D922_C987 + 0
SUBJECT TO
driver_D922: X_D922_C11 + X_D922_C17 + X_D922_C331 + X_D922_C934 + X_D922_C987
 = 1

driver_D88: X_D88_C11 + X_D88_C17 + X_D88_C331 + X_D88_C934 + X_D88_C987 = 1

driver_D32: X_D32_C11 + X_D32_C17 + X_D32_C331 + X_D32_C934 + X_D32_C987 = 1

driver_D8712: X_D8712_C11 + X_D8712_C17 + X_D8712_C331 + X_D8712_C934
 + X_D8712_C987 = 1

driver_D192: X_D192_C11 + X_D192_C17 + X_D192_C331 + X_D192_C934 + X_D192_C987
 = 1

customer_C331: X_D192_C331 + X_D32_C331 + X_D8712_C331 + X_D88_C331
 + X_D922_C331 = 1

customer_C9

In [44]:
# Let's solve the model and make sure it's status is good
model.solve(solver)
print("Status:", LpStatus[model.status])

Status: Optimal


In [45]:
# Here is the solution
for v in model.variables():
    print(v.name, "=", v.varValue)

X_D192_C11 = 0.0
X_D192_C17 = 0.0
X_D192_C331 = 1.0
X_D192_C934 = 0.0
X_D192_C987 = 0.0
X_D32_C11 = 1.0
X_D32_C17 = 0.0
X_D32_C331 = 0.0
X_D32_C934 = 0.0
X_D32_C987 = 0.0
X_D8712_C11 = 0.0
X_D8712_C17 = 0.0
X_D8712_C331 = 0.0
X_D8712_C934 = 0.0
X_D8712_C987 = 1.0
X_D88_C11 = 0.0
X_D88_C17 = 0.0
X_D88_C331 = 0.0
X_D88_C934 = 1.0
X_D88_C987 = 0.0
X_D922_C11 = 0.0
X_D922_C17 = 1.0
X_D922_C331 = 0.0
X_D922_C934 = 0.0
X_D922_C987 = 0.0


In [46]:
print("Total Objective Function Value is = ", value(model.objective))

Total Objective Function Value is =  2443.0
